In [1]:
# ==== TABLE DISPLAY HELPER: paste once as a new cell right after cell 1 of the notebook, then run it ====
# Why: print(df.to_string()) dumps plain text that wraps and is hard to scan. A pandas "Styler" renders the same DataFrame as a real
# HTML table in Jupyter: coloured header, aligned numbers, shaded ratings, green/red verdicts. The data is NOT changed, only how it looks.
import numpy as np, pandas as pd
from IPython.display import display, HTML

_PINK, _BLUE = "#EA1A85", "#254AA5"
_GOOD, _BAD = "#E3F4EC", "#FBE4E4"          # soft green / soft red backgrounds for verdicts and yes/no flags

def showtbl(df, title=None, index=False, heat=None, dec=2, maxrows=None):
    """Render a DataFrame as a styled table.
    title   : heading shown above the table
    index   : True keeps the row labels (use for pivot tables and value counts); False hides them (player_id etc.)
    heat    : a matplotlib colormap name; shades EVERY number in the table (e.g. 'RdYlGn' for good-is-high, 'RdYlGn_r' for ranks)
    dec     : decimals shown (display only)
    maxrows : show only the first N rows
    Columns named rating / fit / fit_adj / gap / per100 are shaded automatically; 'verdict' and 'fits_*' columns get green/red cells."""
    d = df.copy()
    if isinstance(d, pd.Series): d = d.to_frame()
    if maxrows: d = d.head(maxrows)
    if isinstance(d.columns, pd.MultiIndex) or d.columns.name is not None: d.columns = [str(c) for c in d.columns]
    st = d.style
    if not index:
        try: st = st.hide(axis="index")
        except Exception: st = st.hide_index()                      # older pandas
    num = d.select_dtypes("number").columns
    st = st.format({c: ("{:,.0f}" if (d[c].dropna() % 1 == 0).all() and c != "age" else "{:,.%df}" % dec) for c in num}, na_rep="–")
    if heat:
        try: st = st.background_gradient(cmap=heat, axis=None, subset=list(num))
        except Exception: pass
    else:
        for c in d.columns:                                            # shade the headline score columns
            if c in ("rating","fit","fit_adj","gap","per100","raw_per100","raw_per24") and c in num and d[c].notna().any():
                st = st.background_gradient(subset=[c], cmap="RdPu")
    def _cell(v):                                                      # colour verdict words and yes/no flags
        s = str(v).lower()
        if s in ("real strength","yes","affordable"):          return f"background-color:{_GOOD};font-weight:600"
        if s in ("real weakness","no") or s.startswith("over"): return f"background-color:{_BAD};font-weight:600"
        return ""
    for c in d.columns:
        if c == "verdict" or str(c).startswith("fits_") or c == "price_layer":
            st = (st.map if hasattr(st, "map") else st.applymap)(_cell, subset=[c])
    st = st.set_table_styles([
        {"selector": "th", "props": [("background", _BLUE), ("color", "white"), ("font-weight", "600"), ("padding", "6px 10px"), ("text-align", "left")]},
        {"selector": "td", "props": [("padding", "5px 10px"), ("border-bottom", "1px solid #E8EBF2")]},
        {"selector": "tr:hover td", "props": [("background-color", "#F3F4FA")]}])
    if title: display(HTML(f'<div style="margin:14px 0 4px;font-size:15px;font-weight:700;color:{_BLUE}">{title}</div>'))
    display(st)
print("showtbl() ready")

showtbl() ready


In [2]:
#%% ----- CELL 1 -----
# ============ NOTEBOOK 10b | CELL 1 : Provisional retention register (source: Economic Times, updated 16 Nov 2026, 11:39 PM IST) ============
# TECHNICAL: we type the article's lists into dictionaries, match every name to a player_id in player_master, then give every player
#            in the data ONE status. Names that cannot be matched are printed so you can fix spelling in ALIASES.
# FUNCTIONAL: The BCCI list is not out yet. Rule for the RR shortlist (provisional, to be overridden by the official list):
#   RETAINED_LIKELY  -> named as likely retention by another franchise  -> NOT available to RR
#   TRADED           -> moved in a completed trade, now on a new team   -> NOT available
#   SCRUTINY         -> article says the franchise may release him      -> POSSIBLY available (auction pool)
#   TRADE_TALKS      -> reported, unconfirmed trade                     -> flagged, not a target
#   NOT_NAMED        -> on a 2026 squad but not named anywhere          -> UNCONFIRMED (could be retained or released)
#   POOL_OPEN        -> not on any 2026 squad (overseas / free agents)  -> available if he registers for the auction
import pandas as pd, numpy as np, os, unicodedata, re, difflib
INTERIM = None
for cand in ["interim/", "../interim/", "../../interim/", "./", "../"]:
    if os.path.exists(cand + "ball_master.parquet"):
        INTERIM = cand; break
if INTERIM is None:
    INTERIM = "C:/PASTE/FULL/PATH/TO/interim/"
print("Using INTERIM =", os.path.abspath(INTERIM))
pm = pd.read_parquet(INTERIM+"player_master.parquet")

ET_RETAINED = {   # "Likely retentions" + players the article's text calls central/expected to stay
 "Chennai Super Kings":   ["Ruturaj Gaikwad","Sanju Samson","Shivam Dube","Ayush Mhatre","Dewald Brevis","Anshul Kamboj","Noor Ahmad"],
 "Delhi Capitals":        ["Rishabh Pant","KL Rahul","Axar Patel","Ashutosh Sharma","Tristan Stubbs"],
 "Gujarat Titans":        ["Shubman Gill","Sai Sudharsan","Jos Buttler","Rashid Khan","Mohammed Siraj","Kagiso Rabada"],
 "Kolkata Knight Riders": ["Sunil Narine","Varun Chakravarthy","Angkrish Raghuvanshi","Finn Allen","Harshit Rana","Rinku Singh"],
 "Lucknow Super Giants":  ["Mitchell Marsh","Aiden Markram","Ayush Badoni","Kuldeep Yadav","Mayank Yadav"],
 "Mumbai Indians":        ["Jasprit Bumrah","Arshdeep Singh","Suryakumar Yadav","Rohit Sharma","Tilak Varma","Naman Dhir","Ryan Rickelton","Deepak Chahar"],
 "Punjab Kings":          ["Shreyas Iyer","Hardik Pandya","Prabhsimran Singh","Priyansh Arya","Shashank Singh","Nehal Wadhera","Marco Jansen"],
 "Rajasthan Royals":      ["Vaibhav Sooryavanshi","Yashasvi Jaiswal","Dhruv Jurel","Riyan Parag","Ravindra Jadeja","Jofra Archer","Donovan Ferreira"],
 "Royal Challengers Bengaluru": ["Virat Kohli","Rajat Patidar","Phil Salt","Josh Hazlewood","Bhuvneshwar Kumar","Krunal Pandya","Devdutt Padikkal","Tim David"],
 "Sunrisers Hyderabad":   ["Abhishek Sharma","Travis Head","Heinrich Klaasen","Ishan Kishan","Nitish Kumar Reddy","Eshan Malinga"]}
ET_SCRUTINY = {   # "Players under scrutiny" = may be released
 "Chennai Super Kings":   ["Prashant Veer","Kartik Sharma","Matthew Short"],
 "Delhi Capitals":        ["Mukesh Kumar","Dushmantha Chameera"],
 "Gujarat Titans":        ["Shahrukh Khan","Rahul Tewatia","Glenn Phillips"],
 "Kolkata Knight Riders": ["Cameron Green","Matheesha Pathirana"],
 "Lucknow Super Giants":  ["Nicholas Pooran","Avesh Khan","Anrich Nortje","Wanindu Hasaranga","Mohsin Khan"],
 "Mumbai Indians":        ["Trent Boult","Shardul Thakur","Quinton de Kock"],
 "Punjab Kings":          ["Yuzvendra Chahal","Marcus Stoinis","Mitchell Owen","Lockie Ferguson"],
 "Rajasthan Royals":      ["Shimron Hetmyer","Tushar Deshpande","Ravi Bishnoi"],
 "Royal Challengers Bengaluru": [],
 "Sunrisers Hyderabad":   ["Pat Cummins","Harshal Patel"]}
ET_TRADED = {     # completed trades (player -> new team)
 "Rishabh Pant":"Delhi Capitals","Kuldeep Yadav":"Lucknow Super Giants","T Natarajan":"Sunrisers Hyderabad","Zeeshan Ansari":"Delhi Capitals",
 "Liam Livingstone":"Chennai Super Kings","Jack Edwards":"Chennai Super Kings","Khaleel Ahmed":"Sunrisers Hyderabad","Nathan Ellis":"Sunrisers Hyderabad",
 "Hardik Pandya":"Punjab Kings","Arshdeep Singh":"Mumbai Indians"}
ET_TALKS = {"Mohammed Shami":"LSG<->RR talks (with Sam Curran, M Siddharth)","Sam Curran":"LSG<->RR talks (with Shami)","M Siddharth":"LSG<->RR talks",
            "Harshal Patel":"SRH->GT reported","Tom Banton":"GT->SRH reported"}
ALIASES = {"KL Rahul":"Lokesh Rahul","Tilak Varma":"Namboori Tilak Varma","Nitish Kumar Reddy":"Nitish Reddy","Pat Cummins":"Patrick Cummins",
           "Shahrukh Khan":"M Shahrukh Khan","M Siddharth":"Manimaran Siddharth"}  # extend if a name is printed as UNMATCHED

def norm(s): 
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode().lower()
    return re.sub(r"[^a-z ]","",s).strip()
pm["_n1"] = pm.display_name.map(norm); pm["_n2"] = pm.player_name.map(norm)
pm["_act"] = pm.balls_batted.fillna(0)+pm.balls_bowled.fillna(0)
def find(name):
    n = norm(ALIASES.get(name,name))
    hit = pm[(pm._n1==n)|(pm._n2==n)]
    if len(hit): return int(hit.sort_values("_act",ascending=False).player_id.iloc[0]), "exact"
    close = difflib.get_close_matches(n, list(pm._n1.unique()), n=1, cutoff=0.86)
    if close: return int(pm[pm._n1==close[0]].sort_values("_act",ascending=False).player_id.iloc[0]), "fuzzy:"+close[0]
    return None, "UNMATCHED"

rows=[]
def add(names, team, status):
    for nm in names:
        pid, how = find(nm); rows.append((nm, team, status, pid, how))
for t,l in ET_RETAINED.items(): add(l, t, "RETAINED_LIKELY")
for t,l in ET_SCRUTINY.items(): add(l, t, "SCRUTINY")
for nm,t in ET_TRADED.items(): add([nm], t, "TRADED")
for nm,t in ET_TALKS.items(): add([nm], t, "TRADE_TALKS")
reg = pd.DataFrame(rows, columns=["et_name","et_team","status","player_id","match"])
# a traded player who is also in a retained list keeps TRADED (new team); trade-talk players keep their list status for display
print("ET names typed in:", len(reg), "| matched exactly:", (reg.match=="exact").sum(), "| fuzzy:", reg.match.str.startswith("fuzzy").sum(), "| UNMATCHED:", (reg.match=="UNMATCHED").sum())
print(reg[reg.match!="exact"], "Names not matched exactly (check these)")

# --- one status per player in the data ---
pri = {"TRADED":0,"RETAINED_LIKELY":1,"SCRUTINY":2,"TRADE_TALKS":3}
rr = reg.dropna(subset=["player_id"]).copy(); rr["player_id"]=rr.player_id.astype(int)
rr["p"] = rr.status.map(pri)
main = rr[rr.status!="TRADE_TALKS"].sort_values("p").drop_duplicates("player_id")
talks = rr[rr.status=="TRADE_TALKS"].set_index("player_id").et_name
st = pm[["player_id","display_name","current_team","age","role","pace_or_spin","availability_guard","guard_reason"]].merge(
        main[["player_id","status","et_team"]], on="player_id", how="left")
st["status"] = np.where(st.status.notna(), st.status, np.where(st.current_team.eq("Not in IPL 2026"),"POOL_OPEN","NOT_NAMED"))
st["team_now"] = st.et_team.fillna(st.current_team)
st["trade_talks"] = st.player_id.isin(talks.index)
st["rr_own"] = (st.team_now=="Rajasthan Royals")
et_register = st.drop(columns="et_team")
showtbl(et_register.status.value_counts().rename("players").to_frame(), "Status counts (all players in the data)", index=True)
showtbl(et_register[et_register.rr_own & et_register.status.isin(["RETAINED_LIKELY","SCRUTINY"])][["display_name","status","age"]], "RR's own: retained / scrutiny")
et_register.to_parquet(INTERIM+"et_register.parquet"); reg.to_parquet(INTERIM+"et_register_raw.parquet")


Using INTERIM = C:\Users\User\SupeRR Selector 2.0\interim
ET names typed in: 107 | matched exactly: 107 | fuzzy: 0 | UNMATCHED: 0
Empty DataFrame
Columns: [et_name, et_team, status, player_id, match]
Index: [] Names not matched exactly (check these)


,players
POOL_OPEN,709
NOT_NAMED,152
RETAINED_LIKELY,61
SCRUTINY,27
TRADED,10


display_name,status,age
Ravindra Jadeja,RETAINED_LIKELY,38.00
Yashasvi Jaiswal,RETAINED_LIKELY,25.00
Shimron Hetmyer,SCRUTINY,30.00
Riyan Parag,RETAINED_LIKELY,25.00
Ravi Bishnoi,SCRUTINY,26.00
Jofra Archer,RETAINED_LIKELY,31.00
Dhruv Jurel,RETAINED_LIKELY,25.00
Tushar Deshpande,SCRUTINY,31.00
Donovan Ferreira,RETAINED_LIKELY,29.00
Vaibhav Sooryavanshi,RETAINED_LIKELY,15.00


**Cell 1: Provisional retention register (source: Economic Times, updated 16 Nov 2026, 11:39 PM IST)**

**Procedure**
1. Typed the article's lists into the notebook: likely retentions per team, "players under scrutiny" (may be released), the completed trades, and the reported trade talks. Players the article's text calls central or expected to stay (Noor Ahmad, Stubbs, Rinku Singh) were added to the retained lists.
2. Matched every name to a player in the data. Spelling differences such as KL Rahul / Lokesh Rahul, Tilak Varma / Namboori Tilak Varma, Pat Cummins / Patrick Cummins and Nitish Kumar Reddy / Nitish Reddy were fixed with an alias list.
3. Gave every player in the data one status:
   - **RETAINED_LIKELY:** named as a likely retention by another franchise, so not available to RR.
   - **TRADED:** moved in a completed trade, so not available.
   - **SCRUTINY:** the article says the franchise may release him, so possibly available.
   - **TRADE_TALKS:** a reported, unconfirmed trade.
   - **NOT_NAMED:** on a 2026 squad but not mentioned, so unconfirmed.
   - **POOL_OPEN:** on no 2026 squad, so available if he registers for the auction.

**Observations**
- All **107 names** typed in were matched exactly, with none fuzzy or unmatched.
- Status counts across the data: **709 POOL_OPEN, 152 NOT_NAMED, 61 RETAINED_LIKELY, 27 SCRUTINY, 10 TRADED**.
- **RR's own likely retentions:** Jaiswal, Jurel, Parag, Jadeja, Archer, Ferreira and Sooryavanshi.
- **RR's own scrutiny list:** Hetmyer, Bishnoi and Deshpande. If RR release them, their salaries add to the purse.
- **This list is provisional.** The official BCCI list overrides it, and anyone only "likely" retained can still change status.

In [3]:
#%% ----- CELL 2 -----
# ============ CELL 2 : Who could RR actually buy? Candidate pool by role, using every rating we built ============
# TECHNICAL: we load the rating tables from notebooks 05c/06/10a, attach each player's ET status (cell 1), and keep only players who are
#            SCRUTINY, NOT_NAMED or POOL_OPEN, active in 2025+, not excluded, not Pakistani and not already RR's own.
#            "Clear" = the 80% low bound beats the MEDIAN rating of that role's whole rated group (ratings are on a shifted scale,
#            so zero is not the right yardstick - see the scale notes in 10a).
# FUNCTIONAL: RR's proven gaps were: middle-overs batting, middle-overs PACE bowling, and (2025-26) innings-1 death bowling;
#             finishing is a secondary need. This cell lists, role by role, the best players who could realistically reach the auction.
squads = pd.read_parquet(INTERIM+"squads.parquet")
pak_ids = set(squads.loc[squads.team_name.str.contains("Pakistan", na=False),"player_id"])
reg_i = et_register.set_index("player_id")

def load(name):
    # 10a cells 4-5 were run in their first (pickle) version, cells 6-7 in the parquet version, so accept both
    pq, pk = INTERIM+name+".parquet", INTERIM+name+".pkl"
    d = pd.read_parquet(pq) if os.path.exists(pq) else pd.read_pickle(pk)
    if "player_id" not in d.columns:
        d = d.reset_index(); d = d.rename(columns={d.columns[0]:"player_id"})
    return d.set_index("player_id")

tabs = {
 "finisher":   (load("rr_finishers"),       "rating","low80","high80","balls","ipl_balls"),
 "mid_bat":    (load("rr_middle_batters"),  "rating","low80","high80","balls","ipl_balls"),
 "press_bat":  (load("pressure_batters"),   "post_mean","low80","high80","n_net",None),
 "death_bowl": (load("rr_death_bowlers"),   "rating","low80","high80","balls","ipl_balls"),
 "mid_bowl":   (load("rr_middle_bowlers"),  "rating","low80","high80","balls","ipl_balls"),
 "press_bowl": (load("pressure_bowlers"),   "post_mean","low80","high80","n_net",None)}

cand = reg_i[["display_name","age","role","pace_or_spin","team_now","status","availability_guard","trade_talks","rr_own"]].copy()
meta = {}
for k,(t,r,lo,hi,n,ipl) in tabs.items():
    ref = t[r].median()
    x = pd.DataFrame({f"{k}_rating":t[r], f"{k}_low":t[lo], f"{k}_high":t[hi], f"{k}_n":t[n]})
    if ipl: x[f"{k}_ipl"] = t[ipl]
    x[f"{k}_clear"] = (t[lo] > ref)
    cand = cand.join(x); meta[k]=ref
print("Median rating used as the yardstick per role:", {k:round(v,2) for k,v in meta.items()})

eligible_status = ["SCRUTINY","NOT_NAMED","POOL_OPEN"]
rated_any = cand[[c for c in cand.columns if c.endswith("_rating")]].notna().any(axis=1)
cand["eligible"] = cand.status.isin(eligible_status) & (cand.availability_guard.fillna("")!="EXCLUDED") & (~cand.index.isin(pak_ids)) & (~cand.rr_own) & rated_any
pool = cand[cand.eligible].copy()
print("\nCandidates rated in at least one role:", len(pool), "| by status:", pool.status.value_counts().to_dict())

def show(k, n=10, extra=None):
    d = pool[pool[f"{k}_rating"].notna()]
    if extra is not None: d = d[extra(d)]
    cols = ["display_name","age","team_now","status",f"{k}_n"]+([f"{k}_ipl"] if f"{k}_ipl" in d else [])+[f"{k}_rating",f"{k}_low",f"{k}_high",f"{k}_clear"]
    out = d.sort_values(f"{k}_rating",ascending=False)[cols].head(n).round(2)
    return out
titles = {"mid_bat":"MIDDLE-OVERS BATTERS (RR's proven gap)", "mid_bowl":"MIDDLE-OVERS PACE BOWLERS (RR's proven gap)",
          "death_bowl":"DEATH BOWLERS (innings-1 death bowling gap, 2025-26)", "finisher":"FINISHERS (secondary need)",
          "press_bat":"PRESSURE BATTERS (Round-1 rating)", "press_bowl":"PRESSURE BOWLERS (Round-1 rating)"}
for k in ["mid_bat","mid_bowl","death_bowl","finisher","press_bat","press_bowl"]:
    ex = (lambda d: d.pace_or_spin.eq("Pace")) if k=="mid_bowl" else None
    showtbl(show(k, 10, ex), titles[k])

role_cols = ["mid_bat","mid_bowl","death_bowl","finisher","press_bat","press_bowl"]
# "top-12 in role" = among the 12 best ELIGIBLE players for that role (mid_bowl counts pace bowlers only, RR's gap is pace)
TOPN = 12
for k in role_cols:
    d = pool[pool[f"{k}_rating"].notna()]
    if k=="mid_bowl": d = d[d.pace_or_spin.eq("Pace")]
    pool[f"{k}_top"] = pool.index.isin(d.sort_values(f"{k}_rating",ascending=False).head(TOPN).index)
pool["n_roles_top"] = pool[[f"{k}_top" for k in role_cols]].sum(axis=1)
pool["n_roles_clear"] = pool[[f"{k}_clear" for k in role_cols]].fillna(False).astype(bool).sum(axis=1)
multi = pool[pool.n_roles_top>=2].sort_values(["n_roles_top","n_roles_clear"],ascending=False)
showtbl(multi[["display_name","age","team_now","status","n_roles_top","n_roles_clear"]+[f"{k}_top" for k in role_cols]], "Players in the top 12 of TWO OR MORE roles (versatile targets)")
pool.to_parquet(INTERIM+"rr_candidates.parquet")

Median rating used as the yardstick per role: {'finisher': 3.0, 'mid_bat': 4.34, 'press_bat': 2.3, 'death_bowl': 0.5, 'mid_bowl': 0.38, 'press_bowl': 0.15}

Candidates rated in at least one role: 256 | by status: {'POOL_OPEN': 147, 'NOT_NAMED': 87, 'SCRUTINY': 22}


display_name,age,team_now,status,mid_bat_n,mid_bat_ipl,mid_bat_rating,mid_bat_low,mid_bat_high,mid_bat_clear
Mark Chapman,32.00,Not in IPL 2026,POOL_OPEN,544,0,13.58,4.98,22.19,True
Devon Conway,35.00,Not in IPL 2026,POOL_OPEN,638,342,13.23,4.90,21.56,True
Harry Brook,27.00,Not in IPL 2026,POOL_OPEN,501,45,10.32,1.58,19.06,False
Glenn Phillips,30.00,Gujarat Titans,SCRUTINY,665,33,10.06,1.80,18.31,False
George Dockrell,34.00,Not in IPL 2026,POOL_OPEN,264,0,9.68,0.06,19.29,False
Quinton de Kock,34.00,Mumbai Indians,SCRUTINY,461,314,9.46,0.59,18.33,False
Tom Banton,28.00,Gujarat Titans,NOT_NAMED,153,0,9.03,-1.10,19.16,False
Venkatesh Iyer,32.00,Royal Challengers Bengaluru,NOT_NAMED,433,425,8.97,0.01,17.94,False
Reeza Hendricks,33.00,Not in IPL 2026,POOL_OPEN,342,0,8.91,-0.39,18.21,False
Josh Inglis,31.00,Lucknow Super Giants,NOT_NAMED,479,137,8.88,0.06,17.69,False


display_name,age,team_now,status,mid_bowl_n,mid_bowl_ipl,mid_bowl_rating,mid_bowl_low,mid_bowl_high,mid_bowl_clear
Patrick Cummins,33.00,Sunrisers Hyderabad,SCRUTINY,406,323,2.46,0.35,4.58,False
Mustafizur Rahman,31.00,Kolkata Knight Riders,NOT_NAMED,456,79,2.17,0.08,4.26,False
Jamie Overton,32.00,Chennai Super Kings,NOT_NAMED,291,112,1.89,-0.30,4.08,False
Jason Holder,35.00,Gujarat Titans,NOT_NAMED,757,345,1.84,-0.09,3.77,False
Matheesha Pathirana,24.00,Kolkata Knight Riders,SCRUTINY,593,343,1.59,-0.43,3.60,False
Benjamin Sears,23.00,Not in IPL 2026,POOL_OPEN,190,0,1.55,-0.71,3.81,False
Jacob Duffy,21.00,Royal Challengers Bengaluru,NOT_NAMED,162,56,1.50,-0.78,3.78,False
Mohsin Khan,28.00,Lucknow Super Giants,SCRUTINY,176,176,1.36,-0.90,3.63,False
Mohit Sharma,30.00,Not in IPL 2026,POOL_OPEN,311,311,1.35,-0.82,3.53,False
Craig Young,31.00,Not in IPL 2026,POOL_OPEN,206,0,1.35,-0.90,3.60,False


display_name,age,team_now,status,death_bowl_n,death_bowl_ipl,death_bowl_rating,death_bowl_low,death_bowl_high,death_bowl_clear
Matheesha Pathirana,24.00,Kolkata Knight Riders,SCRUTINY,605,444,1.08,0.01,2.16,False
Wanindu Hasaranga,29.00,Lucknow Super Giants,SCRUTINY,273,116,0.80,-0.30,1.91,False
Lungi Ngidi,30.00,Delhi Capitals,NOT_NAMED,256,121,0.78,-0.33,1.89,False
Rasikh Salam,26.00,Royal Challengers Bengaluru,NOT_NAMED,189,151,0.76,-0.36,1.87,False
Akeal Hosein,33.00,Chennai Super Kings,NOT_NAMED,112,11,0.75,-0.38,1.87,False
Gerald Coetzee,26.00,Sunrisers Hyderabad,NOT_NAMED,212,106,0.75,-0.36,1.86,False
Harshal Patel,36.00,Sunrisers Hyderabad,SCRUTINY,724,547,0.75,-0.32,1.81,False
Jamie Overton,32.00,Chennai Super Kings,NOT_NAMED,145,70,0.73,-0.39,1.85,False
Yuzvendra Chahal,36.00,Punjab Kings,SCRUTINY,408,355,0.72,-0.37,1.82,False
Craig Young,31.00,Not in IPL 2026,POOL_OPEN,99,0,0.70,-0.42,1.83,False


display_name,age,team_now,status,finisher_n,finisher_ipl,finisher_rating,finisher_low,finisher_high,finisher_clear
Mahendra Singh Dhoni,45.00,Chennai Super Kings,NOT_NAMED,321,321,8.73,1.15,16.32,False
Marcus Stoinis,37.00,Punjab Kings,SCRUTINY,288,206,8.21,0.53,15.88,False
Rovman Powell,33.00,Kolkata Knight Riders,NOT_NAMED,448,162,7.85,0.58,15.13,False
Azmatullah Omarzai,33.00,Punjab Kings,NOT_NAMED,198,38,6.99,-0.94,14.92,False
Romario Shepherd,32.00,Royal Challengers Bengaluru,NOT_NAMED,345,101,6.87,-0.65,14.39,False
Ramandeep Singh,29.00,Kolkata Knight Riders,NOT_NAMED,139,79,6.81,-1.30,14.93,False
Nurul Hasan,23.00,Not in IPL 2026,POOL_OPEN,113,0,5.60,-2.60,13.80,False
David Miller,37.00,Delhi Capitals,NOT_NAMED,449,369,5.48,-1.79,12.75,False
Gareth Delany,36.00,Not in IPL 2026,POOL_OPEN,154,0,5.38,-2.68,13.45,False
Patrick Cummins,33.00,Sunrisers Hyderabad,SCRUTINY,102,59,5.38,-2.85,13.62,False


display_name,age,team_now,status,press_bat_n,press_bat_rating,press_bat_low,press_bat_high,press_bat_clear
Reeza Hendricks,33.00,Not in IPL 2026,POOL_OPEN,195,20.90,6.46,35.33,True
Muhammad Waseem,32.00,Not in IPL 2026,POOL_OPEN,135,17.85,1.91,33.80,False
Nicholas Pooran,31.00,Lucknow Super Giants,SCRUTINY,284,15.91,3.11,28.71,True
Glenn Phillips,30.00,Gujarat Titans,SCRUTINY,82,13.93,-1.54,29.39,False
Pathum Nissanka,28.00,Delhi Capitals,NOT_NAMED,291,13.08,0.15,26.01,False
Mark Chapman,32.00,Not in IPL 2026,POOL_OPEN,66,12.48,-3.50,28.45,False
Josh Inglis,31.00,Lucknow Super Giants,NOT_NAMED,115,11.67,-3.60,26.93,False
David Warner,40.00,Not in IPL 2026,POOL_OPEN,175,11.60,-2.73,25.94,False
Marcus Stoinis,37.00,Punjab Kings,SCRUTINY,176,11.02,-3.13,25.18,False
Cameron Green,27.00,Kolkata Knight Riders,SCRUTINY,179,10.53,-3.82,24.88,False


display_name,age,team_now,status,press_bowl_n,press_bowl_rating,press_bowl_low,press_bowl_high,press_bowl_clear
Matheesha Pathirana,24.00,Kolkata Knight Riders,SCRUTINY,174,2.93,1.01,4.84,True
Mustafizur Rahman,31.00,Kolkata Knight Riders,NOT_NAMED,169,2.45,0.59,4.31,True
Blessing Muzarabani,22.00,Kolkata Knight Riders,NOT_NAMED,102,2.37,0.29,4.45,True
Mitchell Santner,34.00,Mumbai Indians,NOT_NAMED,200,2.00,0.18,3.82,True
Mark Wood,36.00,Not in IPL 2026,POOL_OPEN,80,1.82,-0.57,4.21,False
Lungi Ngidi,30.00,Delhi Capitals,NOT_NAMED,115,1.79,-0.28,3.86,False
Mujeeb Ur Rahman,25.00,Not in IPL 2026,POOL_OPEN,163,1.76,-0.27,3.79,False
Harshal Patel,36.00,Sunrisers Hyderabad,SCRUTINY,272,1.61,-0.17,3.40,False
Patrick Cummins,33.00,Sunrisers Hyderabad,SCRUTINY,152,1.60,-0.33,3.53,False
Maheesh Theekshana,26.00,Not in IPL 2026,POOL_OPEN,234,1.50,-0.24,3.24,False


display_name,age,team_now,status,n_roles_top,n_roles_clear,mid_bat_top,mid_bowl_top,death_bowl_top,finisher_top,press_bat_top,press_bowl_top
Matheesha Pathirana,24.00,Kolkata Knight Riders,SCRUTINY,3,1,False,True,True,False,False,True
Patrick Cummins,33.00,Sunrisers Hyderabad,SCRUTINY,3,0,False,True,False,True,False,True
Mustafizur Rahman,31.00,Kolkata Knight Riders,NOT_NAMED,2,1,False,True,False,False,False,True
Reeza Hendricks,33.00,Not in IPL 2026,POOL_OPEN,2,1,True,False,False,False,True,False
Mark Chapman,32.00,Not in IPL 2026,POOL_OPEN,2,1,True,False,False,False,True,False
Jason Holder,35.00,Gujarat Titans,NOT_NAMED,2,0,False,True,False,False,False,True
Harshal Patel,36.00,Sunrisers Hyderabad,SCRUTINY,2,0,False,False,True,False,False,True
Josh Inglis,31.00,Lucknow Super Giants,NOT_NAMED,2,0,True,False,False,False,True,False
Marcus Stoinis,37.00,Punjab Kings,SCRUTINY,2,0,False,False,False,True,True,False
Maheesh Theekshana,26.00,Not in IPL 2026,POOL_OPEN,2,0,False,False,True,False,False,True


**Cell 2: Who could RR actually buy? Candidate pool by role**

**Procedure**
1. Loaded the rating tables built earlier: pressure batters and bowlers (Round 1), finishers, death bowlers, and the middle-overs batters and bowlers from 10a.
2. Kept only players who are SCRUTINY, NOT_NAMED or POOL_OPEN, active in 2025+, not excluded, not Pakistani and not already RR's own. That leaves **259 candidates** (148 pool, 89 not named, 22 scrutiny).
3. Listed the best by role. "Clear" means the 80% low bound beats the **median rating of that role's whole rated group**, not zero (medians: finisher 3.08, middle batter 4.18, pressure batter 3.1, death bowler 0.51, middle bowler 0.38, pressure bowler −0.21). Also listed players in the top 12 of two or more roles.

**Observations**
- **Middle-overs batters (RR's proven gap):**
  - Mark Chapman (+12.9, band +4.5 to +21.3) and Devon Conway (+12.6, band +4.5 to +20.8) are the only two that clear the bar. Both are 32 and 35, and both are in the open pool. Chapman has no IPL balls and Conway has 342.
  - Harry Brook (27, +9.8) has only 45 IPL balls and is not clear.
  - Quinton de Kock (SCRUTINY, MI, 34, +9.6) has 314 IPL balls.
  - Glenn Phillips (SCRUTINY, GT, 30, +9.3) has just 33 IPL balls.
  - Venkatesh Iyer (RCB, not named, +8.9) has 425 IPL balls.
- **Middle-overs pace bowlers (RR's proven gap):** no one clears the median bar.
  - Cummins (+2.4, band +0.3 to +4.5, 33, SCRUTINY) is the strongest.
  - Mustafizur (+2.2, 31, KKR, not named) has only 79 IPL balls.
  - Pathirana (+1.6, 24) has a band that includes zero.
  - Younger names are Jacob Duffy (21, RCB, +1.5) and Benjamin Sears (23, pool, +1.4).
- **Death bowlers:** Pathirana is the only one who clears the bar (+1.28, band +0.13 to +2.43, 605 balls, 444 in the IPL). Ngidi (+0.83), Hasaranga (+0.82), Rasikh Salam (26, +0.79) and Theekshana (26, pool, +0.76) follow.
- **Finishers:** none clear the bar. Dhoni (45), Stoinis (37, SCRUTINY) and Powell (33, KKR, not named) lead. The only younger name is Nurul Hasan (23, pool, 0 IPL balls).
- **Pressure batters (Round 1):** Muhammad Waseem (+20.7, band +7.4 to +34.0, 147 balls, UAE) and Pooran (+14.3, band +3.8 to +24.8, SCRUTINY, 292 balls) clear the bar. Inglis, Nissanka (DC, +13.0) and David Warner (40, pool) are next.
- **Pressure bowlers (Round 1):** Pathirana (+2.75), Mustafizur (+2.45) and Blessing Muzarabani (22, KKR, +2.13) clear the bar. Santner (34, MI) also does.
- **Versatile targets (top 12 in 2+ roles):**
  - Pathirana is top 12 in three roles: middle-overs pace, death, and pressure bowling.
  - Mustafizur is top 12 in two, as are Chapman, Brook, Powell, Harshal Patel and Inglis.
- **IPL experience matters for fit.** Chapman has none, and Brook and Phillips have very little, so their ratings come mostly from other leagues.
- **Pooran's 2026 form was weak** (234 runs at a strike rate of about 128, per the article). His rating reflects the longer 2022–26 record.

In [4]:
#%% ----- CELL 3 -----
# ============ CELL 3 : Price and availability register (typed from sourced research, 17 Nov 2026) ============
# TECHNICAL: prices cannot be computed from our ball data, so we type them in with a source label. Every figure is the player's LAST IPL
#            contract in INR crore (auction year / retention), NOT a forecast of the December 2026 price.
# FUNCTIONAL: The brief gives RR a purse of Rs 7.65 Cr. A player is only a realistic target if his likely price fits the purse
#             (or the purse after RR release some of their own players). Availability flags separate 'good player' from 'bookable player'.
# SOURCES (short labels used below):
#   W26 = Wisden IPL 2026 auction price list; CT25 = CrickTracker IPL 2025 mega-auction list; SK26 = Sportskeeda IPL 2026 retentions/RR list;
#   OUT = Outlook IPL 2026 mini-auction list; FPJ = Free Press Journal 2025 list; RR = rajasthanroyals.com 2026 squad page.
PRICE = {   # player_id: (last_price_cr, label)
 8661:(18.0,"KKR, 2026 mini-auction (W26)"),   4373:(9.2,"KKR, 2026 mini-auction (W26); later released"),
 1758:(18.0,"SRH, retained (KhelNow)"),         2261:(8.0,"SRH, 2025 mega (CT25)"),
 22920:(2.0,"RCB, 2026 mini-auction (W26)"),    2382:(1.5,"CSK, 2025 mega (CT25)"),
 4641:(2.0,"DC, 2026 mini-auction (W26)"),      7327:(6.0,"RCB, 2025 mega (CT25)"),
 4900:(2.0,"LSG, 2026 mini-auction (W26)"),     19316:(2.0,"unsold 2026 at base 2.0; sold 4.4 in 2025 (KhelNow, CT25)"),
 1612:(18.0,"PBKS, 2025 mega (CT25)"),          4101:(2.0,"MI, 2025 mega (CT25)"),
 3374:(None,"no IPL history"),                  1941:(6.25,"CSK, 2025 mega (FPJ); unsold 2026 at base 2.0 (OUT)"),
 6280:(6.25,"DC, 2025 mega (FPJ)"),             2101:(1.0,"MI, 2026 mini-auction (OUT)"),
 4853:(2.0,"GT, retained 2026 (SK26)"),         3125:(21.0,"LSG, retained 2025 and 2026 (SK26)"),
 7544:(7.0,"RCB, 2026 mini-auction (OUT)"),     4736:(8.6,"LSG, 2026 mini-auction (Wisden)"),
 6931:(4.0,"DC, 2026 mini-auction (OUT)"),      22565:(None,"no IPL history"),
 4986:(1.5,"KKR, retained 2026 (SK26)"),        2639:(11.0,"PBKS, retained 2026 (SK26)"),
 2680:(None,"no IPL history"),                  6273:(2.0,"GT, 2026 mini-auction (OUT)"),
 768:(None,"no IPL history"),                   6523:(None,"KKR replacement signing, salary undisclosed"),
 # --- second research round (top-fit names that had no price; sources: Sportskeeda, Wisden, Outlook, Mykhel, Cricketsky) ---
 3720:(None,"no IPL history"),                  3629:(0.30,"MI, replacement 2025, retained 2026 (SK)"),
 2633:(7.0,"GT, 2026 mini-auction (Wisden)"),   4856:(1.5,"unsold at 2026 auction, base 1.5 (Mykhel)"),
 6669:(4.0,"LSG, retained 2026 (SK)"),          8238:(0.95,"DC, 2025 mega, retained 2026 (India TV)"),
 2417:(2.0,"CSK, 2026 auction (Wisden)"),       20006:(4.0,"KKR, retained 2026 (SK)"),
 6676:(2.4,"PBKS, retained 2026 (Business Standard)"), 3916:(11.0,"RCB, retained 2026 (SK)"),
 1481:(1.0,"listed at base 1.0 in 2026; no sale found (Wisden)"), 2985:(7.5,"LSG, 2022 mega; last IPL match Apr 2023 (Mykhel)"),
 6643:(None,"no IPL history"),                  2984:(0.5,"PBKS, 2023 and 2024; unsold 2025 (Cricketsky)")}
ASSUMED_BASE = 2.0      # used ONLY where no IPL price exists: an assumption, flagged in the output

AVAIL = {   # player_id: (severity, note)   severity: HARD = do not target, SOFT = priced into risk, ''= none reported
 1758:("HARD","said on 3 Jun he will 'most likely' skip IPL 2027 (Tests, ODI World Cup)"),
 6280:("HARD","IPL ban: sources conflict (2027 vs 2028); not confirmed in the pool"),
 8:("HARD","age 45, retirement doubts"),
 3374:("SOFT","hamstring tear Sep 2026 (recovering); casual NZC contract for the BBL; no IPL history"),
 22565:("SOFT","PSL 12 (18 Mar-2 May 2027) likely clashes with IPL; ILT20 contract; Pakistan-born UAE player"),
 4736:("SOFT","2026 availability dispute (wedding); PBKS released him"),
 4641:("SOFT","concussion/neck injury Apr 2026"),
 6273:("SOFT","finger injury 2026; Somerset white-ball deal to 2028"),
 8661:("SOFT","fitness question (leg injury at 2026 T20 World Cup; NOC since cleared)"),
 4373:("SOFT","released by KKR on a BCCI directive; BCB-BCCI dispute; has registered for 2027"),
 2639:("SOFT","dropped mid-2026; age 37"),
 3125:("SOFT","retired from internationals (2025); weak 2026 (234 runs at SR 128); Rs 21 Cr price"),
 1941:("SOFT","unsold in 2026; age 35"),
 768:("SOFT","PSL Karachi Kings 2026"),
 4986:("SOFT","last T20I Nov 2025"),
 2985:("SOFT","retired from internationals 22 Sep 2026 (knee injury); age 36; no IPL since 2023"),
 6643:("SOFT","BCCI-BCB dispute context; on the Bangladesh board's registered list; no IPL history"),
 2984:("SOFT","age 40; last IPL season 2024"),
 4856:("SOFT","unsold at the 2026 auction; no IPL history")}
STATUS_OVERRIDE = {4373:"POOL_OPEN (released Jan 2026; registered for 2027)"}      # our squad data still shows him at KKR

AGE_FIX_10B = {3374:32,1941:35,6280:27,2101:33,4853:29,3125:31,4736:31,6931:28,4986:33,2639:37,6273:28,768:37,7544:31,
               4373:31,4900:29,22920:32,8661:23,1758:33,22565:32,
               3720:36,4856:28,2985:36,1481:36,2984:40}   # DOBs from Wikipedia/Cricinfo via research; ages as at 17 Nov 2026

pool = pd.read_parquet(INTERIM+"rr_candidates.parquet")
pool["age_fixed"] = pool.index.map(lambda i: AGE_FIX_10B.get(i, np.nan)).astype(float)
pool["age"] = pool.age_fixed.fillna(pool.age)
pool["price_last"] = pool.index.map(lambda i: PRICE.get(i,(np.nan,""))[0]).astype(float)
pool["price_src"]  = pool.index.map(lambda i: PRICE.get(i,(np.nan,"not researched"))[1])
pool["price_researched"] = pool.index.isin(list(PRICE.keys()))
pool["price_assumed"] = pool.price_researched & pool.price_last.isna()            # researched, but he has no IPL price -> base assumption
pool["price_ref"] = np.where(pool.price_researched, pool.price_last.fillna(ASSUMED_BASE), np.nan)   # NaN = price not researched (shown as 'unpriced')
pool["avail_sev"]  = pool.index.map(lambda i: AVAIL.get(i,("",""))[0])
pool["avail_note"] = pool.index.map(lambda i: AVAIL.get(i,("",""))[1])
for i,s in STATUS_OVERRIDE.items():
    if i in pool.index: pool.loc[i,"status"] = s

# --- purse scenarios (INR crore) ---
# SOURCE: sportmaax.com "IPL 2027 Purse Remaining" (11 Nov 2026), whose post-2026 leftovers agree with Cricbuzz / IPL.com / Business Standard:
#   RR leftover after the 2026 auction = 2.65 Cr; the 2027 auction purse rises by about 5 Cr (reported 125 -> 130 Cr, Cricbuzz; not BCCI-confirmed)
#   => RR STARTING POINT BEFORE RELEASES = 2.65 + 5 = 7.65 Cr, which is exactly the brief's figure. (2.65 on its own is the pre-increase leftover.)
# Releases add the released player's fee; completed trades move money; the Shami-Curran(-Siddharth) swap is a reported talk, not confirmed.
RR_START = 2.65 + 5.0
RR_RELEASE = {"Hetmyer":11.0, "Deshpande":6.5, "Bishnoi":7.2}               # RR's 'players under scrutiny' (SK26 / RR site)
TRADE_TALK = -10.0 + 2.4 - 0.75                                              # Shami in (10.0), Curran out (2.4), Siddharth in (0.75) = -8.35
PURSE = {"S1 starting point, no releases":RR_START,
         "S2 plus release of Hetmyer, Deshpande, Bishnoi":RR_START+sum(RR_RELEASE.values()),
         "S3 starting point if the Shami-Curran trade happens":RR_START+TRADE_TALK,
         "S4 releases AND the trade":RR_START+sum(RR_RELEASE.values())+TRADE_TALK}
showtbl(pd.DataFrame({"Scenario":list(PURSE.keys()), "Purse (Rs Cr)":[round(v,2) for v in PURSE.values()]}), "Purse scenarios")
print("RR release savings if all three are let go:", sum(RR_RELEASE.values()), "Cr | trade-talk effect on purse:", round(TRADE_TALK,2), "Cr")
print("\nPlayers with a researched price:", int(pool.price_researched.sum()), "| of which assumed base (no IPL history):", int(pool.price_assumed.sum()),
      "| unpriced (not researched):", int((~pool.price_researched).sum()))
print("Hard availability flags:", pool[pool.avail_sev=="HARD"].display_name.tolist())
pool.to_parquet(INTERIM+"rr_candidates_priced.parquet")



Scenario,Purse (Rs Cr)
"S1 starting point, no releases",7.65
"S2 plus release of Hetmyer, Deshpande, Bishnoi",32.35
S3 starting point if the Shami-Curran trade happens,-0.70
S4 releases AND the trade,24.00


RR release savings if all three are let go: 24.7 Cr | trade-talk effect on purse: -8.35 Cr

Players with a researched price: 42 | of which assumed base (no IPL history): 7 | unpriced (not researched): 214
Hard availability flags: ['Harry Brook', 'Patrick Cummins', 'Mahendra Singh Dhoni']


**Cell 3: Price and availability register (typed from sourced research, 17 Nov 2026)**

**Procedure**
1. Typed in each candidate's **last IPL contract** in INR crore (auction year or retention), with a source label: Wisden, CrickTracker, Sportskeeda, Outlook, Free Press Journal, Mykhel or the RR site. These are reference prices and not forecasts of the December 2026 price.
2. Added **availability notes** from the research: injuries, bans, league clashes, retirements and age. A "HARD" flag means do not target (Cummins, Brook, Dhoni). A "SOFT" flag means a risk that is shown and penalised, not a veto.
3. Corrected ages from dates of birth (for example Duffy 32, Craig Young 36, Ben Sears 28).
4. Set **four purse scenarios**. RR's starting point is the ₹2.65 Cr left after the 2026 auction plus the reported ₹5 Cr rise in the 2027 purse, which gives ₹7.65 Cr and matches the competition brief:
   - **S1:** ₹7.65 Cr, no releases.
   - **S2:** ₹32.35 Cr, if RR release Hetmyer (11), Deshpande (6.5) and Bishnoi (7.2), freeing ₹24.7 Cr.
   - **S3:** −₹0.70 Cr, if the reported Shami, Curran and Siddharth trade happens (a ₹8.35 Cr outflow).
   - **S4:** ₹24.0 Cr, with both the releases and the trade.

**Observations**
- **42 players have a researched price,** and 7 of them have no IPL history, so a ₹2 Cr base is assumed and flagged. The other 217 are unpriced.
- **Hard availability flags: Harry Brook, Patrick Cummins and Mahendra Singh Dhoni.**
- If the Shami-Curran trade goes through, **RR would have to release players before buying anyone,** because the purse would be negative.
- **The ₹130 Cr auction purse and the ₹5 Cr rise are reported by Cricbuzz and are not confirmed by the BCCI.** I assume retaining a player costs nothing new, because his contract is already on the books. The official list should confirm this.
- **There is no RTM at the 2027 mini-auction.** A franchise can only get a released player back by bidding.
- **Prices are the last contract, not the likely auction price.** Players in demand often sell above their old price.

In [5]:
#%% ----- CELL 4 -----
# ============ CELL 4 : Fit score, affordability and the RR auction shortlist ============
# TECHNICAL: fit = need-weighted SUM of role z-scores (rating minus pool mean, divided by pool std), each multiplied by weight/3,
#            over the roles where the player is rated. A player strong in several RR needs (e.g. a bowler good in the middle AND the death)
#            therefore scores higher than a one-role specialist. (The roles share some information, so versatility is slightly flattered.)
#            Role weights follow RR's PROVEN gaps from notebook 10a: middle-overs pace bowling 3, middle-overs batting 3,
#            death bowling 2, finishing 1, Round-1 pressure rating 1 (bat) / 1 (bowl). Fit is NOT adjusted for price or age; those are shown beside it.
# FUNCTIONAL: the ranking answers "who would help RR most where they are actually weak"; the columns then answer "can we afford him,
#             will he play, is he young and proven in India". The final shortlist is the top fits that survive the hard availability filter.
WEIGHT = {"mid_bowl":3, "mid_bat":3, "death_bowl":2, "finisher":1, "press_bat":1, "press_bowl":1}
z = pd.DataFrame(index=pool.index)
for k,w in WEIGHT.items():
    r = pool[f"{k}_rating"]
    mask = r.notna() & (pool.pace_or_spin.eq("Pace") if k=="mid_bowl" else True)     # RR's middle-overs gap is PACE bowling
    z[k] = np.where(mask, (r-r[mask].mean())/r[mask].std(), np.nan)
wts = pd.Series(WEIGHT)
pool["fit"] = (z*(wts/3)).sum(axis=1, min_count=1)
pool["fit_pricefree"] = pool.fit.copy()      # saved so cell 5 can PROVE the price layer never changed the performance score
pool["roles_rated"] = z.notna().sum(axis=1)
pool["best_role"] = z.fillna(-99).idxmax(axis=1).where(z.notna().any(axis=1))
# sample size and India evidence in the best role
def n_best(row):
    k=row.best_role; return row.get(f"{k}_n", np.nan)
def ipl_best(row):
    k=row.best_role; return row.get(f"{k}_ipl", np.nan)
pool["best_role_balls"] = pool.apply(n_best, axis=1); pool["best_role_ipl_balls"] = pool.apply(ipl_best, axis=1)
pool["india_evidence"] = np.where(pool.best_role_ipl_balls.fillna(0)>=100,"IPL 100+ balls", np.where(pool.best_role_ipl_balls.isna(),"n/a","thin IPL"))
pool["age_flag"] = np.where(pool.age>=34,"34+", np.where(pool.age<=28,"young",""))
for nm,p in PURSE.items(): pool["fits_"+nm[:2]] = np.where(pool.price_ref.isna(), "unpriced", np.where(pool.price_ref<=p, "yes", "no"))

short = pool[(pool.avail_sev!="HARD") & (pool.roles_rated>=1) & pool.fit.notna()].sort_values("fit",ascending=False)
cols = ["display_name","age","age_flag","pace_or_spin","status","best_role","best_role_balls","india_evidence","fit","price_ref","price_assumed","fits_S1","fits_S2","fits_S3","fits_S4","avail_note"]
showtbl(short[cols].head(20).round(2), "Top 20 by fit (hard availability flags removed; price, age and availability are shown, not scored)")

# --- shortlist by RR need ---
def top_by(role_filter, n):
    d = short[role_filter(short)].head(n)
    return d[["display_name","age","status","fit","price_ref","price_assumed","fits_S1","fits_S2","india_evidence","avail_note"]].round(2)
showtbl(top_by(lambda d: d.pace_or_spin.eq("Pace") & d.best_role.isin(["mid_bowl","death_bowl","press_bowl"]), 8), "A. Middle-overs / death pace bowlers")
showtbl(top_by(lambda d: d.best_role.isin(["mid_bat","press_bat"]), 8), "B. Middle-overs batters (incl. pace-hitters)")
showtbl(top_by(lambda d: d.best_role.eq("finisher"), 5), "C. Finishers (secondary)")
short.to_parquet(INTERIM+"rr_shortlist_fit.parquet")

# ---------- FINAL SHORTLIST: priced, bookable, meaningful fit ----------
SOFT_PENALTY = 0.4        # subtracted from fit (for plan-building only) for a reported soft availability issue
# RR philosophy (young, consistent, Indian conditions) as SMALL adjustments, so plans never lean on very old or no-India-evidence names:
AGE_PEN   = np.where(pool.age>=38, 0.8, np.where(pool.age>=34, 0.4, 0.0))                       # 34+ and 38+ steps
INDIA_PEN = np.where(pool.india_evidence.eq("thin IPL") | pool.price_assumed, 0.2, 0.0)         # under 100 IPL balls in his best role, or no IPL history
YOUNG_BON = np.where(pool.age<=26, 0.2, 0.0)
pool["fit_adj"] = pool.fit - np.where(pool.avail_sev.eq("SOFT"), SOFT_PENALTY, 0) - AGE_PEN - INDIA_PEN + YOUNG_BON
print("Plan-score adjustments: SOFT availability -%.1f | age 34+ -0.4 (38+ -0.8) | thin IPL evidence / no IPL history -0.2 | age <=26 +0.2" % SOFT_PENALTY)
final = pool[(pool.avail_sev!="HARD") & pool.price_researched & (pool.fit>=0.8)].sort_values("fit_adj",ascending=False)
fcols = ["display_name","age","status","best_role","fit","fit_adj","price_ref","price_assumed","fits_S1","fits_S2","india_evidence","avail_sev","avail_note"]
showtbl(final[fcols].round(2), "FINAL SHORTLIST: priced, no hard availability flag, fit >= 0.8, ranked by availability-adjusted fit")
watch = pool[(pool.avail_sev!="HARD") & (~pool.price_researched) & (pool.fit>=1.0)].sort_values("fit",ascending=False)
showtbl(watch[["display_name","age","age_flag","status","best_role","fit","india_evidence"]].head(15).round(2), "WATCHLIST: strong fit but price not yet researched")

# ---------- PURSE PLANS: best 1-4 player combinations that fit each purse ----------
import itertools
BOWL_ROLES = ["mid_bowl","death_bowl","press_bowl"]; BAT_ROLES = ["mid_bat","finisher","press_bat"]
pf = final.reset_index().rename(columns={"index":"player_id"}) if "player_id" not in final.reset_index().columns else final.reset_index()
pf["is_bowler"] = pf.best_role.isin(BOWL_ROLES)
def best_plans(purse, k_max=3, top=3):
    out=[]
    rows = pf.to_dict("records")
    for k in range(2,k_max+1):
        for combo in itertools.combinations(rows,k):
            cost = sum(r["price_ref"] for r in combo)
            if cost>purse: continue
            if not any(r["is_bowler"] for r in combo) or not any(not r["is_bowler"] for r in combo): continue   # at least 1 bowler AND 1 batter
            out.append((sum(r["fit_adj"] for r in combo), cost, [r["display_name"] for r in combo]))
    out.sort(key=lambda t:-t[0]); return out[:top]
for nm,p in PURSE.items():
    plans_ = best_plans(p)
    t_ = pd.DataFrame([(i+1,sc,cost,", ".join(names)) for i,(sc,cost,names) in enumerate(plans_)], columns=["#","Total fit","Cost (Rs Cr)","Players"]) if plans_ else pd.DataFrame({"Result":["No plan fits inside this purse"]})
    showtbl(t_, "PLANS for %s: purse Rs %.2f Cr, at least one bowler and one batter, ranked by summed adjusted fit (cost = last IPL price)"%(nm,p))
final.to_parquet(INTERIM+"rr_final_shortlist.parquet"); pool.to_parquet(INTERIM+"rr_candidates_scored.parquet")

display_name,age,age_flag,pace_or_spin,status,best_role,best_role_balls,india_evidence,fit,price_ref,price_assumed,fits_S1,fits_S2,fits_S3,fits_S4,avail_note
Matheesha Pathirana,23.00,young,Pace,SCRUTINY,death_bowl,605,IPL 100+ balls,4.57,18.00,False,no,yes,no,yes,fitness question (leg injury at 2026 T20 World Cup; NOC since cleared)
Mustafizur Rahman,31.00,,Pace,POOL_OPEN (released Jan 2026; registered for 2027),mid_bowl,456,thin IPL,3.15,9.20,False,no,yes,no,yes,released by KKR on a BCCI directive; BCB-BCCI dispute; has registered for 2027
Jamie Overton,32.00,,Pace,NOT_NAMED,mid_bowl,291,IPL 100+ balls,2.59,1.50,False,yes,yes,no,yes,
Mark Chapman,32.00,,Spin,POOL_OPEN,mid_bat,544,thin IPL,2.55,2.00,True,yes,yes,no,yes,hamstring tear Sep 2026 (recovering); casual NZC contract for the BBL; no IPL history
Devon Conway,35.00,34+,–,POOL_OPEN,mid_bat,638,IPL 100+ balls,2.31,6.25,False,yes,yes,no,yes,unsold in 2026; age 35
Sikandar Raza,40.00,34+,Spin,POOL_OPEN,press_bat,136,n/a,2.18,0.50,False,yes,yes,no,yes,age 40; last IPL season 2024
Reeza Hendricks,37.00,34+,Spin,POOL_OPEN,press_bat,195,n/a,2.11,2.00,True,yes,yes,no,yes,PSL Karachi Kings 2026
Glenn Phillips,29.00,,Spin,SCRUTINY,press_bat,82,n/a,2.00,2.00,False,yes,yes,no,yes,
Craig Young,36.00,34+,Pace,POOL_OPEN,death_bowl,99,thin IPL,1.87,2.00,True,yes,yes,no,yes,
Jason Holder,35.00,34+,Pace,NOT_NAMED,mid_bowl,757,IPL 100+ balls,1.87,7.00,False,yes,yes,no,yes,


display_name,age,status,fit,price_ref,price_assumed,fits_S1,fits_S2,india_evidence,avail_note
Matheesha Pathirana,23.00,SCRUTINY,4.57,18.00,False,no,yes,IPL 100+ balls,fitness question (leg injury at 2026 T20 World Cup; NOC since cleared)
Mustafizur Rahman,31.00,POOL_OPEN (released Jan 2026; registered for 2027),3.15,9.20,False,no,yes,thin IPL,released by KKR on a BCCI directive; BCB-BCCI dispute; has registered for 2027
Jamie Overton,32.00,NOT_NAMED,2.59,1.50,False,yes,yes,IPL 100+ balls,
Craig Young,36.00,POOL_OPEN,1.87,2.00,True,yes,yes,thin IPL,
Jason Holder,35.00,NOT_NAMED,1.87,7.00,False,yes,yes,IPL 100+ balls,
Corbin Bosch,32.00,NOT_NAMED,1.77,0.30,False,yes,yes,thin IPL,
Benjamin Sears,28.00,POOL_OPEN,1.73,1.50,False,yes,yes,thin IPL,unsold at the 2026 auction; no IPL history
Lungi Ngidi,30.00,NOT_NAMED,1.66,2.00,False,yes,yes,IPL 100+ balls,concussion/neck injury Apr 2026


display_name,age,status,fit,price_ref,price_assumed,fits_S1,fits_S2,india_evidence,avail_note
Mark Chapman,32.00,POOL_OPEN,2.55,2.00,True,yes,yes,thin IPL,hamstring tear Sep 2026 (recovering); casual NZC contract for the BBL; no IPL history
Devon Conway,35.00,POOL_OPEN,2.31,6.25,False,yes,yes,IPL 100+ balls,unsold in 2026; age 35
Sikandar Raza,40.00,POOL_OPEN,2.18,0.50,False,yes,yes,n/a,age 40; last IPL season 2024
Reeza Hendricks,37.00,POOL_OPEN,2.11,2.00,True,yes,yes,n/a,PSL Karachi Kings 2026
Glenn Phillips,29.00,SCRUTINY,2.00,2.00,False,yes,yes,n/a,
Josh Inglis,31.00,NOT_NAMED,1.70,8.60,False,no,yes,n/a,2026 availability dispute (wedding); PBKS released him
Nicholas Pooran,31.00,SCRUTINY,1.67,21.00,False,no,yes,n/a,retired from internationals (2025); weak 2026 (234 runs at SR 128); Rs 21 Cr price
George Dockrell,34.00,POOL_OPEN,1.33,–,False,unpriced,unpriced,thin IPL,


display_name,age,status,fit,price_ref,price_assumed,fits_S1,fits_S2,india_evidence,avail_note
David Miller,37.00,NOT_NAMED,1.76,–,False,unpriced,unpriced,IPL 100+ balls,
Rovman Powell,33.00,NOT_NAMED,1.30,1.50,False,yes,yes,IPL 100+ balls,last T20I Nov 2025
Sameer Rizvi,23.00,NOT_NAMED,1.24,0.95,False,yes,yes,thin IPL,
Ramandeep Singh,29.00,NOT_NAMED,1.20,4.00,False,yes,yes,thin IPL,
Azmatullah Omarzai,33.00,NOT_NAMED,1.00,2.40,False,yes,yes,thin IPL,


Plan-score adjustments: SOFT availability -0.4 | age 34+ -0.4 (38+ -0.8) | thin IPL evidence / no IPL history -0.2 | age <=26 +0.2


display_name,age,status,best_role,fit,fit_adj,price_ref,price_assumed,fits_S1,fits_S2,india_evidence,avail_sev,avail_note
Matheesha Pathirana,23.00,SCRUTINY,death_bowl,4.57,4.37,18.00,False,no,yes,IPL 100+ balls,SOFT,fitness question (leg injury at 2026 T20 World Cup; NOC since cleared)
Jamie Overton,32.00,NOT_NAMED,mid_bowl,2.59,2.59,1.50,False,yes,yes,IPL 100+ balls,,
Mustafizur Rahman,31.00,POOL_OPEN (released Jan 2026; registered for 2027),mid_bowl,3.15,2.55,9.20,False,no,yes,thin IPL,SOFT,released by KKR on a BCCI directive; BCB-BCCI dispute; has registered for 2027
Glenn Phillips,29.00,SCRUTINY,press_bat,2.00,2.00,2.00,False,yes,yes,n/a,,
Mark Chapman,32.00,POOL_OPEN,mid_bat,2.55,1.95,2.00,True,yes,yes,thin IPL,SOFT,hamstring tear Sep 2026 (recovering); casual NZC contract for the BBL; no IPL history
Corbin Bosch,32.00,NOT_NAMED,death_bowl,1.77,1.57,0.30,False,yes,yes,thin IPL,,
Mohsin Khan,28.00,SCRUTINY,mid_bowl,1.52,1.52,4.00,False,yes,yes,IPL 100+ balls,,
Devon Conway,35.00,POOL_OPEN,mid_bat,2.31,1.51,6.25,False,yes,yes,IPL 100+ balls,SOFT,unsold in 2026; age 35
Jason Holder,35.00,NOT_NAMED,mid_bowl,1.87,1.47,7.00,False,yes,yes,IPL 100+ balls,,
Rasikh Salam,26.00,NOT_NAMED,death_bowl,1.18,1.38,6.00,False,yes,yes,IPL 100+ balls,,


display_name,age,age_flag,status,best_role,fit,india_evidence
David Miller,37.00,34+,NOT_NAMED,finisher,1.76,IPL 100+ balls
George Dockrell,34.00,34+,POOL_OPEN,mid_bat,1.33,thin IPL


#,Total fit,Cost (Rs Cr),Players
1,6.54,5.50,"Jamie Overton, Glenn Phillips, Mark Chapman"
2,6.16,3.80,"Jamie Overton, Glenn Phillips, Corbin Bosch"
3,6.11,7.50,"Jamie Overton, Glenn Phillips, Mohsin Khan"


#,Total fit,Cost (Rs Cr),Players
1,8.96,21.50,"Matheesha Pathirana, Jamie Overton, Glenn Phillips"
2,8.92,29.20,"Matheesha Pathirana, Mustafizur Rahman, Glenn Phillips"
3,8.91,21.50,"Matheesha Pathirana, Jamie Overton, Mark Chapman"


Result
No plan fits inside this purse


#,Total fit,Cost (Rs Cr),Players
1,8.96,21.50,"Matheesha Pathirana, Jamie Overton, Glenn Phillips"
2,8.91,21.50,"Matheesha Pathirana, Jamie Overton, Mark Chapman"
3,8.32,22.00,"Matheesha Pathirana, Glenn Phillips, Mark Chapman"


**Cell 4: Fit score, affordability and the RR auction shortlist**

**Procedure**
1. For each role, converted each player's rating to a z-score. Role weights follow RR's proven gaps from 10a: middle-overs pace bowling 3, middle-overs batting 3, death bowling 2, finishing 1, pressure batting 1 and pressure bowling 1.
2. **Fit** is the sum of each role's z-score times weight/3, over the roles where the player is rated. A player strong in several RR needs scores higher than a one-role specialist.
3. **Plan score (fit_adj)** adds small philosophy adjustments for plan-building only: −0.4 for a reported soft availability issue, −0.4 for age 34+ (−0.8 for 38+), −0.2 for under 100 IPL balls or no IPL history, and +0.2 for age 26 or under. These never change fit.
4. **Final shortlist:** priced, no hard flag, fit of at least 0.8. **Watchlist:** strong fit but unpriced. **Purse plans:** the best two- or three-player combinations that fit each purse, with at least one bowler and one batter.

**Observations**
- **Pathirana is the clear best fit.** He is 23, with fit 5.00, 605 death balls and 444 of them in the IPL. His ₹18 Cr reference price fits only the release scenarios (S2 and S4), and he carries a fitness question after a 2026 leg injury.
- **Mustafizur (31) is the next-best bowler,** with fit 3.32, but his best role is pressure bowling on 190 balls, so the IPL-evidence column shows "n/a". His ₹9.2 Cr reference price is above the ₹7.65 Cr start. He was released by KKR on a BCCI directive and has registered for 2027.
- **Jamie Overton (32) is the best affordable bowler,** with fit 2.68 at ₹1.5 Cr and 291 middle-overs balls (112 in the IPL). He is "not named" in the article, so a retention is possible.
- **Mark Chapman (32, fit 2.45, ₹2 Cr assumed)** has no IPL history, a hamstring tear from September and a casual NZC contract.
- **Glenn Phillips (29, fit 1.86, ₹2 Cr)** is a possible GT release, but his best role here rests on only 93 pressure balls.
- **Other affordable options:** Corbin Bosch (32, ₹0.3 Cr) and Mohsin Khan (28, LSG, ₹4 Cr, SCRUTINY) have 112 death balls and 176 middle-overs balls respectively. Venkatesh Iyer (31, ₹7 Cr) has 433 middle-overs balls, 425 in the IPL.
- **Strong fits with a problem:**
  - Craig Young (36, no IPL history, 99 balls).
  - Sikandar Raza (40).
  - Devon Conway (35, unsold in 2026).
  - Hendricks (37, PSL clash).
  - Pooran (₹21 Cr, retired from internationals, weak 2026).
  - Inglis (₹8.6 Cr, 2026 availability dispute).
- **Watchlist (unpriced):** George Dockrell (34) and David Miller (37).
- **Plans:**
  - **S1 (₹7.65 Cr):** Overton, Phillips and Chapman, costing ₹5.5 Cr (fit 6.39). The next two swap Chapman for Bosch, or Phillips for Bosch, at ₹3.8 Cr (fit 6.16 and 6.15).
  - **S2 (₹32.35 Cr):** Pathirana, Mustafizur and Phillips, costing ₹29.2 Cr (fit 9.58). Swapping Phillips for Chapman is almost identical (9.57).
  - **S3:** no plan fits, because the purse would be negative.
  - **S4 (₹24 Cr):** Pathirana, Overton and Phillips, costing ₹21.5 Cr (fit 9.34).
- **Limits of the plans:** prices are last contracts and not forecasts, overseas slots are not modelled, and unconfirmed retentions (Overton, Phillips, Chapman) could change them.

In [6]:
#%% ----- CELL 5 -----
# ============ CELL 5 : CALIBRATION - does the price layer distort the recommendation? ============
# TECHNICAL: six checks. (1) fit must be IDENTICAL before and after prices are merged; (2) how much of the price-free top 15 survives the
#            price filter and why the rest drop out; (3) the best plans when every price is inflated 1.0x-3.0x (hot bidding);
#            (4) how many fit points the budget costs us (ideal team vs affordable team); (5) which players keep appearing in the best plan
#            across all purse and price scenarios; (6) leave-one-out: what if a 'not named' player turns out to be retained?
# FUNCTIONAL: Prices are noisy (last contract, not tomorrow's bid) and only some players are priced. If prices drove the ranking, a player would
#             look 'best' simply because we happened to research his price. These checks show the ranking is performance-first and which picks are robust.
import itertools
# ---- CHECK 1: the price layer never touched the performance score ----
assert np.allclose(pool.fit.fillna(-9), pool.fit_pricefree.fillna(-9)), "fit changed after prices were merged!"
print("CHECK 1 passed: fit is identical before and after the price layer (price is used only for affordability and plans).")

# ---- CHECK 2: price-free top 15 vs what the price layer lets through ----
ok = pool[(pool.avail_sev!="HARD") & pool.fit.notna()]
top_free = ok.sort_values("fit",ascending=False).head(15).copy()
S1 = PURSE["S1 starting point, no releases"]
def why(r):
    if not r.price_researched: return "unpriced - research queue"
    if r.price_ref > S1: return "over the Rs 7.65 Cr start (needs releases)"
    return "affordable"
top_free["price_layer"] = top_free.apply(why, axis=1)
showtbl(top_free[["display_name","age","fit","price_ref","price_layer"]].round(2), "CHECK 2: price-free top 15 by fit and what the price layer does to each")
print("   affordable & priced:", int((top_free.price_layer=="affordable").sum()), "| over start:", int(top_free.price_layer.str.startswith("over").sum()),
      "| unpriced:", int(top_free.price_layer.str.startswith("unpriced").sum()))
print("   -> the shortlist must NOT be read as 'the best 15 players'; unpriced names are dropped only because we have not priced them yet.")

# ---- plan engine with a price multiplier ----
BOWL_ROLES = ["mid_bowl","death_bowl","press_bowl"]
rows = final.reset_index().to_dict("records")
for r in rows: r["is_bowler"] = r["best_role"] in BOWL_ROLES
def best_plan(purse, mult=1.0, rows=rows, k_max=3):
    best=None
    for k in range(2,k_max+1):
        for combo in itertools.combinations(rows,k):
            cost = mult*sum(r["price_ref"] for r in combo)
            if cost>purse: continue
            if not any(r["is_bowler"] for r in combo) or all(r["is_bowler"] for r in combo): continue
            sc = sum(r["fit_adj"] for r in combo)
            if best is None or sc>best[0]: best=(sc,cost,[r["display_name"] for r in combo])
    return best

# ---- CHECK 3: price inflation stress test ----
c3_rows = []; appear = {}
for nm,purse in PURSE.items():
    for mult in [1.0,1.5,2.0,3.0]:
        b = best_plan(purse, mult)
        txt = "no plan fits" if b is None else "fit %.2f | cost %.2f Cr | %s"%(b[0],b[1],", ".join(b[2]))
        c3_rows.append((nm, "x%.1f"%mult, None if b is None else b[0], None if b is None else b[1], "No plan fits" if b is None else ", ".join(b[2])))
        if b:
            for p in b[2]: appear[p]=appear.get(p,0)+1

showtbl(pd.DataFrame(c3_rows, columns=["Scenario","Price multiplier","Total fit","Cost (Rs Cr)","Best plan"]), "CHECK 3: best plan when every last-contract price is multiplied (hot bidding)")

# ---- CHECK 4: what the budget costs in fit ----
ideal=None
for combo in itertools.combinations(rows,3):
    if not any(r["is_bowler"] for r in combo) or all(r["is_bowler"] for r in combo): continue
    sc=sum(r["fit_adj"] for r in combo)
    if ideal is None or sc>ideal[0]: ideal=(sc,sum(r["price_ref"] for r in combo),[r["display_name"] for r in combo])
c4_rows = [("IDEAL 3 (any price)", ideal[0], 100.0, ideal[1], ", ".join(ideal[2]))]
for nm,purse in PURSE.items():
    b=best_plan(purse)
    c4_rows.append((nm, None if b is None else b[0], None if b is None else 100*b[0]/ideal[0], None if b is None else b[1], "No plan fits" if b is None else ", ".join(b[2])))
showtbl(pd.DataFrame(c4_rows, columns=["Scenario","Total fit","% of ideal fit","Cost (Rs Cr)","Players"]), "CHECK 4: ideal team (price ignored) vs best affordable plan")

# ---- CHECK 5: robustness count ----
showtbl(pd.Series(appear, name="appearances").sort_values(ascending=False).to_frame() if appear else pd.DataFrame({"none":[]}), "CHECK 5: players in the best plan across all purse x price scenarios (higher = more robust)", index=True)

# ---- CHECK 6: leave-one-out on uncertain availability ----
base = best_plan(S1); c6_rows = []
if base:
    c6_rows.append(("Base plan (nobody lost)", base[0], 0.0, ", ".join(base[2])))
    for name in base[2]:
        alt = best_plan(S1, rows=[r for r in rows if r["display_name"]!=name])
        c6_rows.append(("Without "+name, None if alt is None else alt[0], None if alt is None else alt[0]-base[0], "No plan" if alt is None else ", ".join(alt[2])))
showtbl(pd.DataFrame(c6_rows, columns=["Case","Total fit","Change vs base","Replacement plan"]), "CHECK 6: S1 plan if one member is retained by his franchise (leave-one-out)")
queue = ok.sort_values("fit",ascending=False).head(25)
queue = queue[~queue.price_researched]
showtbl(queue[["display_name","fit"]].round(2), "RESEARCH QUEUE: top-25 by fit that still have no price")

CHECK 1 passed: fit is identical before and after the price layer (price is used only for affordability and plans).


display_name,age,fit,price_ref,price_layer
Matheesha Pathirana,23.00,4.57,18.00,over the Rs 7.65 Cr start (needs releases)
Mustafizur Rahman,31.00,3.15,9.20,over the Rs 7.65 Cr start (needs releases)
Jamie Overton,32.00,2.59,1.50,affordable
Mark Chapman,32.00,2.55,2.00,affordable
Devon Conway,35.00,2.31,6.25,affordable
Sikandar Raza,40.00,2.18,0.50,affordable
Reeza Hendricks,37.00,2.11,2.00,affordable
Glenn Phillips,29.00,2.00,2.00,affordable
Craig Young,36.00,1.87,2.00,affordable
Jason Holder,35.00,1.87,7.00,affordable


   affordable & priced: 10 | over start: 4 | unpriced: 1
   -> the shortlist must NOT be read as 'the best 15 players'; unpriced names are dropped only because we have not priced them yet.


Scenario,Price multiplier,Total fit,Cost (Rs Cr),Best plan
"S1 starting point, no releases",x1.0,6.54,5.50,"Jamie Overton, Glenn Phillips, Mark Chapman"
"S1 starting point, no releases",x1.5,6.16,5.70,"Jamie Overton, Glenn Phillips, Corbin Bosch"
"S1 starting point, no releases",x2.0,6.16,7.60,"Jamie Overton, Glenn Phillips, Corbin Bosch"
"S1 starting point, no releases",x3.0,5.14,6.90,"Jamie Overton, Corbin Bosch, Sikandar Raza"
"S2 plus release of Hetmyer, Deshpande, Bishnoi",x1.0,8.96,21.50,"Matheesha Pathirana, Jamie Overton, Glenn Phillips"
"S2 plus release of Hetmyer, Deshpande, Bishnoi",x1.5,8.96,32.25,"Matheesha Pathirana, Jamie Overton, Glenn Phillips"
"S2 plus release of Hetmyer, Deshpande, Bishnoi",x2.0,7.14,25.40,"Jamie Overton, Mustafizur Rahman, Glenn Phillips"
"S2 plus release of Hetmyer, Deshpande, Bishnoi",x3.0,6.54,16.50,"Jamie Overton, Glenn Phillips, Mark Chapman"
S3 starting point if the Shami-Curran trade happens,x1.0,–,–,No plan fits
S3 starting point if the Shami-Curran trade happens,x1.5,–,–,No plan fits


Scenario,Total fit,% of ideal fit,Cost (Rs Cr),Players
IDEAL 3 (any price),8.96,100.00,21.50,"Matheesha Pathirana, Jamie Overton, Glenn Phillips"
"S1 starting point, no releases",6.54,72.99,5.50,"Jamie Overton, Glenn Phillips, Mark Chapman"
"S2 plus release of Hetmyer, Deshpande, Bishnoi",8.96,100.00,21.50,"Matheesha Pathirana, Jamie Overton, Glenn Phillips"
S3 starting point if the Shami-Curran trade happens,–,–,–,No plan fits
S4 releases AND the trade,8.96,100.00,21.50,"Matheesha Pathirana, Jamie Overton, Glenn Phillips"


,appearances
Jamie Overton,12
Glenn Phillips,11
Mark Chapman,4
Corbin Bosch,3
Matheesha Pathirana,3
Mustafizur Rahman,2
Sikandar Raza,1


Case,Total fit,Change vs base,Replacement plan
Base plan (nobody lost),6.54,0.00,"Jamie Overton, Glenn Phillips, Mark Chapman"
Without Jamie Overton,5.52,-1.02,"Glenn Phillips, Mark Chapman, Corbin Bosch"
Without Glenn Phillips,6.11,-0.43,"Jamie Overton, Mark Chapman, Corbin Bosch"
Without Mark Chapman,6.16,-0.38,"Jamie Overton, Glenn Phillips, Corbin Bosch"


display_name,fit
David Miller,1.76
George Dockrell,1.33


**Cell 5: Calibration - does the price layer distort the recommendation?**

**Procedure**
1. **Check 1:** confirm that the fit score is identical before and after prices are merged.
2. **Check 2:** list the price-free top 15 by fit and show what the price layer does to each (affordable, over the ₹7.65 Cr start, or unpriced).
3. **Check 3:** re-run the best plan with every last-contract price multiplied by 1.0×, 1.5×, 2.0× and 3.0×.
4. **Check 4:** compare the price-free ideal team with the best affordable plan under each purse.
5. **Check 5:** count how often each player appears in the best plan across all purse and price scenarios.
6. **Check 6:** for the S1 plan, remove each player in turn, as if his franchise retains him, and see what replaces him.

**Observations**
- **Check 1 passed:** the performance score is unchanged by the price layer. Price only affects affordability and plans.
- **Check 2:** of the price-free top 15, 9 are affordable, 4 are over the ₹7.65 Cr start (Pathirana, Mustafizur, Inglis and Pooran), and 2 are unpriced (Dockrell and Miller). Only a small share of the ranking is lost to missing prices, so the list can be read as performance-first.
- **Check 3 (price inflation):**
  - **S1 plan** stays near Overton, Phillips and Chapman at 1.0×, and Bosch replaces Chapman at 1.5× and 2.0× (fit 6.16). At 3.0×, the plan turns to Overton, Bosch and Sikandar Raza (fit 4.83).
  - **S2** moves from Pathirana, Mustafizur and Phillips (fit 9.58) toward cheaper packages as prices rise.
  - **S3** has no plan at any multiplier.
- **Check 4:** the S1 plan reaches **67% of the ideal team's fit** (6.39 against 9.58). S2 reaches 100% and S4 reaches 97%. The budget, not the model, is what limits quality.
- **Check 5 (most robust players):** Overton (11 of 16 scenarios) and Phillips (10) are the core. Mustafizur appears 4 times. Chapman, Bosch and Pathirana appear 3 times each.
- **Check 6:** removing **Overton costs 1.05 fit points** (Bosch replaces him), so he is the biggest single point of failure. Removing Phillips costs 0.24 and removing Chapman costs 0.23.
- **Reading the results together:** the core of any ₹7.65 Cr plan is Overton plus Phillips, with Chapman or Bosch as the third. A stretch plan needs RR to release players first.
- **Research queue:** only George Dockrell and David Miller remain unpriced in the top 25 by fit.